In [ ]:
import pandas as pd
import numpy as np
import os
import duckdb

#### Leitura e transformação dos dados

In [ ]:
# Definindo a estrutura de caminhos relativos do projeto
BRONZE_DIR = "../data/bronze"
SILVER_DIR = "../data/silver"

In [ ]:
def ler_nomes_arquivos_bronze():
    """
    Lê os nomes dos arquivos na pasta bronze e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(BRONZE_DIR) if os.path.isfile(os.path.join(BRONZE_DIR, f))]

In [ ]:
lista_arquivos = ler_nomes_arquivos_bronze()

In [ ]:
def transformar_csv_para_parquet(lista_arquivos):
    """
    Função para transformar um arquivo CSV em Parquet.
    
    Parâmetros:
    lista_arquivos (list): Lista de nomes dos arquivos CSV de entrada.
    """

    for arquivo in lista_arquivos:
        # Lendo o arquivo CSV
        df = pd.read_csv(os.path.join(BRONZE_DIR, arquivo))
        
        # Definindo o nome do arquivo Parquet de saída
        nome_arquivo_parquet = os.path.splitext(arquivo)[0] + '.parquet'
        
        # Salvando o DataFrame como Parquet
        df.to_parquet(os.path.join(SILVER_DIR, nome_arquivo_parquet), index=False)
        
        print(f"Arquivo {arquivo} transformado")


In [ ]:
transformar_csv_para_parquet(lista_arquivos)

In [ ]:
def ler_nomes_arquivos_silver():
    """
    Lê os nomes dos arquivos na pasta silver e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(SILVER_DIR) if os.path.isfile(os.path.join(SILVER_DIR, f))]

In [ ]:
ler_nomes_arquivos_silver()

In [ ]:
operacoes = pd.read_parquet(os.path.join(SILVER_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet'))

operacoes.info()

#### Tratamento de valores nulos

In [ ]:
def verificar_valores_nulos(df):
    """Verifica se um DataFrame contém valores nulos.

    Parâmetros:
        df (DataFrame): O DataFrame a ser verificado.

    Retorna:
        False | DataFrame: False se não houver nulos; DataFrame com análise se houver.
    """
    total_nulos = df.isnull().sum().sort_values(ascending=False)
    total_nulos = total_nulos[total_nulos > 0]

    if total_nulos.empty:
        return False

    total_nulos_percent = ((total_nulos / df.shape[0]) * 100).round(2)
    return pd.DataFrame({'Total Nulls': total_nulos, '%': total_nulos_percent})

In [ ]:
verificar_valores_nulos(operacoes)

In [ ]:
#categorias existentes na coluna tipo_excepcionalidade
operacoes["tipo_excepcionalidade"].value_counts(dropna=False)

**Regra de negócio:** Se _tipo_excepcionalidade_
 está nulo, significa que a operação seguiu o fluxo padrão (sem exceção). Então, podemos realizar a classificação 0 = não, e 1 = sim.

In [ ]:
#Criando uma nova coluna para indicar se a operação possui ou não excepcionalidade
operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)

In [ ]:
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

In [ ]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

**Regra de Négocio:** Se o CNPJ da instituição financeira credenciada estiver nulo, podemos entender que a operação foi realizada diretamente com o BNDES, sem intermediação de uma instituição financeira. Portanto, vamos preencher esses valores nulos com a string "OPERAÇÃO DIRETA".

In [ ]:
operacoes["cnpj_instituicao_financeira_credenciada"] = operacoes["cnpj_instituicao_financeira_credenciada"].fillna("0000000000000.0")
operacoes["nome_instituicao_financeira_credenciada"] = operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
print(f"{operacoes[['cnpj_instituicao_financeira_credenciada']].dtypes}")

In [ ]:
#Verficando colunas restantes para tratamento de valores nulos
operacoes.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
print(f"{operacoes[['cnpj_cliente']].dtypes}")

In [ ]:
#preenchendo os valores nulos
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna("000000000000.0")
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

In [ ]:
#removendo colunas cnae desnecessárias
operacoes = operacoes.drop(columns=["classe_cnae","subclasse_cnae","grupo_cnae","divisao_cnae","secao_cnae"])

In [ ]:
verificar_valores_nulos(operacoes)

In [ ]:
#Verificando os tipos de dados das colunas
operacoes.dtypes

In [ ]:
#consultando colunas de data
cols_data = [c for c in operacoes.columns if "data" in c]
cols_data
operacoes[cols_data].dtypes

In [ ]:
#Ajuste dos tipos de dados das colunas de data
operacoes[cols_data] = operacoes[cols_data].apply(pd.to_datetime, errors='coerce')
operacoes[cols_data].dtypes

#### Enriquecimento de dados

In [ ]:
operacoes["ano_contratacao"] = operacoes["data_contratacao"].dt.year
operacoes["mes_contratacao"] = operacoes["data_contratacao"].dt.month

In [ ]:
silver_operacoes = operacoes.copy()

total = silver_operacoes[["valor_contratado", "valor_desembolsado"]].sum().round(2)
total_silver= total.apply(lambda x: f"{x:,.2f}")
total_silver

In [ ]:
# Ler os dados brutos
raw = pd.read_csv(
    os.path.join(BRONZE_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv'),
    dtype=str
)

cols = ["valor_contratado", "valor_desembolsado"]

In [ ]:
total = raw[cols].apply(pd.to_numeric, errors="coerce").sum().round(2)

total__rawformatado = total.apply(lambda x: f"{x:,.2f}")
total_raw_formatado